# 🍟 Snackspert - Website Recensies naar Google Docs

Dit notebook verzamelt alle recensies van **snackspert.nl/restaurant** en maakt per recensie een apart Google Docs-bestand aan in een Google Drive-map.

## Hoe te gebruiken
1. Voer elke cel uit door op het **▶ play-knopje** te klikken (of druk `Shift+Enter`)
2. Bij stap 2 moet je inloggen met je Google-account
3. Stap 3 t/m 5 doen het werk!

---

## Stap 1: Installeer benodigde packages
Dit hoef je maar één keer te doen per sessie.

In [ ]:
!pip install -q requests beautifulsoup4 google-api-python-client google-auth-httplib2 google-auth-oauthlib
print("\n\u2705 Alle packages ge\u00efnstalleerd!")

## Stap 2: Log in met je Google-account
Er verschijnt een pop-up om in te loggen. Dit geeft het notebook toegang tot je Google Drive en Google Docs.

In [ ]:
from google.colab import auth
auth.authenticate_user()

from google.auth import default
creds, _ = default()

from googleapiclient.discovery import build
docs_service = build('docs', 'v1', credentials=creds)
drive_service = build('drive', 'v3', credentials=creds)

print("\u2705 Ingelogd en verbonden met Google Drive & Docs!")

## Stap 3: Instellingen

In [ ]:
# === INSTELLINGEN ===

# Naam van de Google Drive-map waar de documenten in komen
# (wordt automatisch aangemaakt als deze niet bestaat)
DRIVE_MAP_NAAM = "Snackspert Recensies"

# Maximaal aantal recensies ophalen (0 = alles)
MAX_RECENSIES = 0

print(f"\u2705 Instellingen geladen:")
print(f"   Drive-map: {DRIVE_MAP_NAAM}")
print(f"   Max recensies: {'alles' if MAX_RECENSIES == 0 else MAX_RECENSIES}")

## Stap 4: Recensies ophalen van snackspert.nl

Dit haalt alle restaurants op via de WordPress REST API en scrapet vervolgens elke restaurantpagina voor de recensietekst en sterren.

**Dit kan 10-30 minuten duren** afhankelijk van het aantal restaurants (~700).

Als een pagina meerdere recensies bevat (meerdere alinea's met sterren), worden die **apart** opgeslagen.

In [ ]:
import re
import time
import requests
from dataclasses import dataclass
from bs4 import BeautifulSoup


@dataclass
class Recensie:
    """Een enkele Snackspert-recensie."""
    naam: str
    adres: str
    tekst: str
    sterren: float
    sterren_tekst: str
    afbeelding_url: str
    pagina_url: str
    deel_nummer: int = 0  # 0 = enige recensie, 1/2/3 = meerdere delen


def tel_sterren_in_tekst(tekst: str) -> tuple:
    """Tel het aantal sterren (emoji's) in een stuk tekst."""
    volle = len(re.findall(r'\u2b50\ufe0f?', tekst))
    halve = 1 if ('\u00bd' in tekst or '1/2' in tekst) else 0
    totaal = volle + (0.5 if halve else 0)
    sterren_str = '\u2b50' * volle + ('\u00bd' if halve else '')
    return totaal, sterren_str


def scrape_restaurant_pagina(url: str, sessie: requests.Session) -> list:
    """
    Scrape een individuele restaurantpagina.
    Retourneert een LIJST van recensies (meestal 1, soms meer).
    """
    resp = sessie.get(url, timeout=30)
    resp.raise_for_status()
    soup = BeautifulSoup(resp.text, 'html.parser')

    # Naam
    naam_el = soup.select_one('.restaurantBlock .bigTitle')
    naam = naam_el.get_text(strip=True) if naam_el else ''

    # Adres
    adres_el = soup.select_one('.restaurantBlock .innerAddress')
    adres = adres_el.get_text(strip=True) if adres_el else ''

    # Afbeelding
    img_el = soup.select_one('.restaurantBlock .innerImage')
    afbeelding_url = ''
    if img_el and img_el.get('style'):
        match = re.search(r"url\('([^']+)'\)", img_el['style'])
        if match:
            afbeelding_url = match.group(1)

    # Recensietekst - per <p> apart bekijken
    tekst_el = soup.select_one('.restaurantBlock .text')
    if not tekst_el:
        return [{
            'naam': naam, 'adres': adres, 'tekst': '',
            'sterren': 0, 'sterren_tekst': '',
            'afbeelding_url': afbeelding_url, 'deel_nummer': 0,
        }]

    # Zoek alle <p> tags met sterren erin
    paragrafen = tekst_el.find_all('p')
    delen_met_sterren = []

    for p in paragrafen:
        p_tekst = p.get_text(strip=True)
        sterren, sterren_str = tel_sterren_in_tekst(p_tekst)
        if sterren > 0:
            delen_met_sterren.append({
                'tekst': p_tekst,
                'sterren': sterren,
                'sterren_tekst': sterren_str,
            })

    # Als er geen sterren gevonden zijn, neem de hele tekst
    if not delen_met_sterren:
        volledige_tekst = tekst_el.get_text(strip=True)
        return [{
            'naam': naam, 'adres': adres, 'tekst': volledige_tekst,
            'sterren': 0, 'sterren_tekst': '',
            'afbeelding_url': afbeelding_url, 'deel_nummer': 0,
        }]

    # Als er maar 1 deel is met sterren, neem de hele tekst + die sterren
    if len(delen_met_sterren) == 1:
        volledige_tekst = tekst_el.get_text(strip=True)
        return [{
            'naam': naam, 'adres': adres, 'tekst': volledige_tekst,
            'sterren': delen_met_sterren[0]['sterren'],
            'sterren_tekst': delen_met_sterren[0]['sterren_tekst'],
            'afbeelding_url': afbeelding_url, 'deel_nummer': 0,
        }]

    # Meerdere delen met sterren = meerdere recensies op 1 pagina
    resultaten = []
    for i, deel in enumerate(delen_met_sterren, 1):
        resultaten.append({
            'naam': naam, 'adres': adres, 'tekst': deel['tekst'],
            'sterren': deel['sterren'],
            'sterren_tekst': deel['sterren_tekst'],
            'afbeelding_url': afbeelding_url,
            'deel_nummer': i,
        })
    return resultaten


# --- Stap 1: Alle restaurant-URLs ophalen via de WP REST API ---
print("\U0001f50d Stap 4a: Alle restaurants ophalen via de REST API...\n")

sessie = requests.Session()
sessie.headers.update({
    'User-Agent': 'Mozilla/5.0 (X11; CrOS x86_64) AppleWebKit/537.36 Chrome/124.0 Safari/537.36'
})

alle_restaurants = []
pagina = 1
while True:
    api_url = f"https://snackspert.nl/wp-json/wp/v2/restaurant?per_page=100&page={pagina}"
    resp = sessie.get(api_url, timeout=30)
    if resp.status_code != 200:
        break
    data = resp.json()
    if not data:
        break
    for item in data:
        alle_restaurants.append({
            'id': item['id'],
            'naam': item['title']['rendered'],
            'url': item['link'],
        })
    totaal = int(resp.headers.get('X-WP-Total', 0))
    totaal_paginas = int(resp.headers.get('X-WP-TotalPages', 0))
    print(f"  Pagina {pagina}/{totaal_paginas} - {len(alle_restaurants)}/{totaal} restaurants")
    if pagina >= totaal_paginas:
        break
    pagina += 1
    time.sleep(0.5)

print(f"\n\u2705 {len(alle_restaurants)} restaurants gevonden!\n")

# --- Stap 2: Elke restaurantpagina scrapen ---
print("\U0001f50d Stap 4b: Elke restaurantpagina scrapen voor recensie + sterren...\n")

if MAX_RECENSIES > 0:
    alle_restaurants = alle_restaurants[:MAX_RECENSIES]
    print(f"   (Beperkt tot {MAX_RECENSIES} restaurants)\n")

recensies = []
fouten = []
for i, rest in enumerate(alle_restaurants, 1):
    print(f"  [{i}/{len(alle_restaurants)}] {rest['naam'][:50]}...", end=" ")
    try:
        delen = scrape_restaurant_pagina(rest['url'], sessie)
        for deel in delen:
            recensie = Recensie(
                naam=deel['naam'] or rest['naam'],
                adres=deel['adres'],
                tekst=deel['tekst'],
                sterren=deel['sterren'],
                sterren_tekst=deel['sterren_tekst'],
                afbeelding_url=deel['afbeelding_url'],
                pagina_url=rest['url'],
                deel_nummer=deel['deel_nummer'],
            )
            recensies.append(recensie)
        if len(delen) > 1:
            print(f"\u2705 {len(delen)} recensies gevonden (apart opgeslagen)")
        else:
            print(f"\u2705 {delen[0]['sterren']} sterren")
    except Exception as e:
        fouten.append({'naam': rest['naam'], 'url': rest['url'], 'fout': str(e)})
        print(f"\u274c {e}")
    if i % 10 == 0:
        time.sleep(1)
    else:
        time.sleep(0.3)

print(f"\n\u2705 {len(recensies)} recensies opgehaald van {len(alle_restaurants)} restaurants!")
if fouten:
    print(f"\u26a0\ufe0f {len(fouten)} fouten opgetreden")

# Controle: geen enkele score boven 5
boven_5 = [r for r in recensies if r.sterren > 5]
if boven_5:
    print(f"\n\u26a0\ufe0f Waarschuwing: {len(boven_5)} recensies met >5 sterren gevonden (fout in brontekst)")
    for r in boven_5:
        print(f"   - {r.naam}: {r.sterren} sterren")
else:
    print(f"\u2705 Alle scores zijn 5 of lager - geen dubbeltelling!")

## Stap 5: Recensies opslaan als Google Docs
Per recensie wordt een apart document aangemaakt in de map op je Google Drive.

Bij restaurants met meerdere recensies krijgt elk document een volgnummer (bijv. "Snackspert - Restaurant (deel 1)").

In [ ]:
import html as html_module

# --- Google Drive-map zoeken of aanmaken ---
print(f"\U0001f4c1 Map '{DRIVE_MAP_NAAM}' zoeken of aanmaken...\n")

query = f"name = '{DRIVE_MAP_NAAM}' and mimeType = 'application/vnd.google-apps.folder' and trashed = false"
result = drive_service.files().list(q=query, fields='files(id, name)').execute()
folders = result.get('files', [])

if folders:
    folder_id = folders[0]['id']
    print(f"\u2705 Bestaande map gevonden: {DRIVE_MAP_NAAM}")
else:
    folder_metadata = {
        'name': DRIVE_MAP_NAAM,
        'mimeType': 'application/vnd.google-apps.folder'
    }
    folder = drive_service.files().create(body=folder_metadata, fields='id').execute()
    folder_id = folder['id']
    print(f"\u2705 Nieuwe map aangemaakt: {DRIVE_MAP_NAAM}")

print(f"   Map-ID: {folder_id}")
print(f"   \U0001f517 https://drive.google.com/drive/folders/{folder_id}\n")


def maak_recensie_doc(recensie, folder_id):
    """Maak een Google Docs-bestand aan voor \u00e9\u00e9n recensie."""
    clean_naam = html_module.unescape(recensie.naam)
    if recensie.deel_nummer > 0:
        doc_title = f"Snackspert - {clean_naam} (deel {recensie.deel_nummer})"
    else:
        doc_title = f"Snackspert - {clean_naam}"

    # Leeg document aanmaken
    doc = docs_service.documents().create(body={'title': doc_title}).execute()
    doc_id = doc['documentId']

    # Verplaats naar de juiste map
    drive_service.files().update(
        fileId=doc_id,
        addParents=folder_id,
        removeParents='root',
        fields='id, parents'
    ).execute()

    # Document vullen met inhoud
    sections = []

    # Titel
    titel = clean_naam
    if recensie.deel_nummer > 0:
        titel += f" (deel {recensie.deel_nummer})"
    sections.append({'text': f"{titel}\n", 'style': 'HEADING_1'})

    # Sterren
    sterren_display = f"{recensie.sterren} van 5 sterren"
    if recensie.sterren_tekst:
        sterren_display = f"{recensie.sterren_tekst} ({recensie.sterren}/5)"
    sections.append({'text': f"Beoordeling: {sterren_display}\n\n", 'style': 'NORMAL_TEXT'})

    # Metadata
    meta_lines = []
    if recensie.adres:
        meta_lines.append(f"Adres: {recensie.adres}")
    meta_lines.append(f"Website: {recensie.pagina_url}")
    sections.append({'text': '\n'.join(meta_lines) + '\n\n', 'style': 'NORMAL_TEXT'})

    # Recensie tekst
    sections.append({'text': 'Recensie\n', 'style': 'HEADING_2'})
    sections.append({'text': (recensie.tekst or '(Geen tekst)') + '\n\n', 'style': 'NORMAL_TEXT'})

    # Afbeelding link
    if recensie.afbeelding_url:
        sections.append({'text': 'Afbeelding\n', 'style': 'HEADING_2'})
        sections.append({'text': recensie.afbeelding_url + '\n', 'style': 'NORMAL_TEXT'})

    # Requests opbouwen
    requests_list = []
    index = 1
    for section in sections:
        text = section['text']
        requests_list.append({
            'insertText': {
                'location': {'index': index},
                'text': text,
            }
        })
        if section['style'] != 'NORMAL_TEXT':
            requests_list.append({
                'updateParagraphStyle': {
                    'range': {'startIndex': index, 'endIndex': index + len(text)},
                    'paragraphStyle': {'namedStyleType': section['style']},
                    'fields': 'namedStyleType',
                }
            })
        index += len(text)

    if requests_list:
        docs_service.documents().batchUpdate(
            documentId=doc_id, body={'requests': requests_list}
        ).execute()

    return f"https://docs.google.com/document/d/{doc_id}/edit"


# --- Alle recensies verwerken ---
print(f"\U0001f4dd {len(recensies)} documenten aanmaken...\n")

resultaten = []
for i, recensie in enumerate(recensies, 1):
    clean_naam = html_module.unescape(recensie.naam)
    label = clean_naam[:50]
    if recensie.deel_nummer > 0:
        label += f" (deel {recensie.deel_nummer})"
    print(f"  [{i}/{len(recensies)}] {label}...", end=" ")
    try:
        url = maak_recensie_doc(recensie, folder_id)
        resultaten.append({'naam': clean_naam, 'deel': recensie.deel_nummer, 'sterren': recensie.sterren, 'url': url})
        print(f"\u2705")
    except Exception as e:
        resultaten.append({'naam': clean_naam, 'deel': recensie.deel_nummer, 'sterren': recensie.sterren, 'url': None, 'error': str(e)})
        print(f"\u274c {e}")

# Samenvatting
gelukt = [r for r in resultaten if r.get('url')]
mislukt = [r for r in resultaten if not r.get('url')]

print(f"\n{'='*50}")
print(f"\u2705 KLAAR!")
print(f"{'='*50}")
print(f"  Totaal:    {len(recensies)} recensies")
print(f"  Gelukt:    {len(gelukt)}")
if mislukt:
    print(f"  Mislukt:   {len(mislukt)}")
print(f"\n\U0001f4c2 Open je Google Drive-map:")
print(f"   https://drive.google.com/drive/folders/{folder_id}")

## Stap 6 (optioneel): Bekijk een overzicht
Bekijk een tabel met alle aangemaakte documenten, gesorteerd op sterren.

In [ ]:
import pandas as pd

df = pd.DataFrame([
    {
        'Restaurant': r['naam'][:40] + (f" (deel {r['deel']})" if r.get('deel', 0) > 0 else ''),
        'Sterren': r['sterren'],
        'Status': '\u2705' if r.get('url') else '\u274c',
        'Google Docs Link': r.get('url', r.get('error', '-'))
    }
    for r in resultaten
])

df_sorted = df.sort_values('Sterren', ascending=False)

print(f"\U0001f4ca Overzicht van alle {len(resultaten)} recensies:\n")
print(f"Gemiddelde score: {df['Sterren'].mean():.1f} sterren")
print(f"Hoogste score:    {df['Sterren'].max()} sterren")
print(f"Laagste score:    {df['Sterren'].min()} sterren")
print()
df_sorted

---
## Stap 7: Repareer bestaande Google Docs (eenmalig)

**Voer alleen Stap 1 en deze cel uit.** Je hoeft geen andere stappen te draaien - deze cel regelt zelf de Google-login en zoekt de map.

Dit zoekt alle documenten in je "Snackspert Recensies" map waar de beoordeling hoger is dan 5 sterren, en corrigeert de score automatisch door alleen de eerste sterren-groep te pakken.

In [ ]:
import re

# === Eigen setup - geen andere stappen nodig behalve Stap 1 ===
from google.colab import auth
auth.authenticate_user()
from google.auth import default
creds, _ = default()
from googleapiclient.discovery import build
docs_service = build('docs', 'v1', credentials=creds)
drive_service = build('drive', 'v3', credentials=creds)

# Pas dit aan als je map anders heet
DRIVE_MAP_NAAM = "Snackspert Recensies"

print("\U0001f527 Bestaande documenten controleren en repareren...\n")

# Zoek de map
query = f"name = '{DRIVE_MAP_NAAM}' and mimeType = 'application/vnd.google-apps.folder' and trashed = false"
result = drive_service.files().list(q=query, fields='files(id, name)').execute()
folders = result.get('files', [])
if not folders:
    print("\u274c Map niet gevonden! Controleer of de mapnaam klopt.")
else:
    folder_id = folders[0]['id']
    print(f"\u2705 Map gevonden: {DRIVE_MAP_NAAM}\n")

    # Alle documenten in de map ophalen
    alle_docs = []
    page_token = None
    while True:
        q = f"'{folder_id}' in parents and mimeType = 'application/vnd.google-apps.document' and trashed = false"
        resp = drive_service.files().list(
            q=q, fields='nextPageToken, files(id, name)',
            pageSize=100, pageToken=page_token
        ).execute()
        alle_docs.extend(resp.get('files', []))
        page_token = resp.get('nextPageToken')
        if not page_token:
            break

    print(f"   {len(alle_docs)} documenten gevonden in de map\n")

    gerepareerd = 0
    al_goed = 0
    fouten = 0
    for i, doc_info in enumerate(alle_docs, 1):
        try:
            # Lees het document
            doc = docs_service.documents().get(documentId=doc_info['id']).execute()
            content = doc.get('body', {}).get('content', [])

            # Zoek de volledige tekst
            full_text = ''
            for element in content:
                if 'paragraph' in element:
                    for elem in element['paragraph'].get('elements', []):
                        if 'textRun' in elem:
                            full_text += elem['textRun']['content']

            # Zoek de beoordelingsregel
            match = re.search(r'Beoordeling:.*?([\d.]+)/5', full_text)
            if not match:
                continue

            oude_score = float(match.group(1))
            if oude_score <= 5:
                al_goed += 1
                continue  # Score is al correct

            # Score is >5, we moeten repareren
            print(f"  [{gerepareerd + 1}] {doc_info['name'][:50]} - score {oude_score} > 5, repareren...", end=" ")

            # Zoek de recensietekst en vind de EERSTE groep sterren
            recensie_match = re.search(r'Recensie\n(.+?)(?:\n\n|Afbeelding)', full_text, re.DOTALL)
            if not recensie_match:
                print("\u26a0\ufe0f geen recensietekst gevonden")
                continue

            recensie_tekst = recensie_match.group(1)

            # Zoek alle sterren-clusters in de tekst
            star_pattern = r'[\u2b50\ufe0f]+\u00bd?'
            clusters = list(re.finditer(star_pattern, recensie_tekst))

            if not clusters:
                print("\u26a0\ufe0f geen sterren in recensietekst")
                continue

            # Neem de eerste cluster = eerste beoordeling
            eerste_cluster = clusters[0].group()
            volle = len(re.findall(r'\u2b50\ufe0f?', eerste_cluster))
            halve = 1 if '\u00bd' in eerste_cluster else 0
            nieuwe_score = volle + (0.5 if halve else 0)

            if nieuwe_score > 5:
                nieuwe_score = 5  # Cap op 5

            # Zoek de exacte positie van de beoordelingsregel in het document
            beoordeling_start = None
            beoordeling_end = None
            for element in content:
                if 'paragraph' in element:
                    for elem in element['paragraph'].get('elements', []):
                        if 'textRun' in elem:
                            run_text = elem['textRun']['content']
                            if 'Beoordeling:' in run_text and beoordeling_start is None:
                                beoordeling_start = elem['startIndex']
                                beoordeling_end = elem['endIndex']

            if beoordeling_start is None:
                print("\u26a0\ufe0f positie niet gevonden")
                continue

            # Maak nieuwe beoordelingstekst
            nieuwe_sterren_str = '\u2b50' * int(nieuwe_score) + ('\u00bd' if nieuwe_score % 1 else '')
            nieuwe_tekst = f"Beoordeling: {nieuwe_sterren_str} ({nieuwe_score}/5)\n"

            # Update het document
            docs_service.documents().batchUpdate(
                documentId=doc_info['id'],
                body={'requests': [
                    {'deleteContentRange': {'range': {'startIndex': beoordeling_start, 'endIndex': beoordeling_end}}},
                    {'insertText': {'location': {'index': beoordeling_start}, 'text': nieuwe_tekst}},
                ]}
            ).execute()

            print(f"\u2705 {oude_score} \u2192 {nieuwe_score}")
            gerepareerd += 1

        except Exception as e:
            print(f"\u274c Fout bij {doc_info['name'][:40]}: {e}")
            fouten += 1

    print(f"\n{'='*50}")
    print(f"\u2705 Klaar!")
    print(f"   {len(alle_docs)} documenten gecontroleerd")
    print(f"   {al_goed} waren al correct")
    print(f"   {gerepareerd} gerepareerd")
    if fouten:
        print(f"   {fouten} fouten")